# Gold Layer: Customer Dimension (DLT)

**Pattern**: SCD Type 2 tracking with Metadata-driven Quality Checks  
**Source**: Silver tables (CRM and ERP)

This notebook builds the final customer dimension, tracking all historical changes and applying dynamic data quality rules.

In [0]:
import dlt
from pyspark.sql.functions import col, md5, concat_ws, current_date

# 1. Helper to load dynamic expectations from metadata
def get_rules(table_name):
    """Loads quality rules from the central metadata table"""
    try:
        rules_df = spark.read.table("workspace.metadata.quality_rules").where(col("table_name") == table_name)
        return {row.rule_name: row.condition for row in rules_df.collect()}
    except:
        # Fallback if metadata table isn't ready
        return {"valid_id": "customer_id IS NOT NULL"}

# 2. Join multiple silver sources into a single view
@dlt.view
def gold_dim_customers_source():
    ci = dlt.read("silver_crm_customers_cdc")
    # Note: These ERP tables should also be defined in your DLT pipeline
    ca = dlt.read("silver_erp_cust_cdc") 
    la = dlt.read("silver_erp_loc_cdc")
    
    return (
        ci.alias("ci")
        .join(ca.alias("ca"), col("ci.customer_number") == col("ca.customer_number"), "left")
        .join(la.alias("la"), col("ci.customer_number") == col("la.customer_number"), "left")
        .select(
            col("ci.customer_id"),
            col("ci.customer_number"),
            col("ci.first_name"),
            col("ci.last_name"),
            col("la.country"),
            col("ci.marital_status"),
            col("ci.gender"),
            col("ca.birth_date").alias("birthdate"),
            col("ci.created_date").alias("create_date"),
            col("ci.ingestion_timestamp")
        )
    )

# 3. Apply SCD Type 2 tracking and Quality Checks
dlt.create_streaming_table(
    name="gold_dim_customers",
    comment="Final Customer Dimension with full history (SCD Type 2)"
)

dlt.apply_changes(
    target="gold_dim_customers",
    source="gold_dim_customers_source",
    keys=["customer_id"],
    sequence_by=col("ingestion_timestamp"),
    stored_as_scd_type=2, # DLT handles effective_from, effective_to, and is_current automatically
    expect_all_or_drop=get_rules("gold_dim_customers")
)